In [134]:
# importing modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder


df = pd.read_csv('data/Iris.csv')
print("Data Shape: ", df.shape)

df.drop('Id', axis=1, inplace=True) # remove id column
df.head()

Data Shape:  (150, 6)


,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


Data Analysis

In [135]:
# feature matrix
x_orig = df.iloc[:, 1:-1].values

# data labels
y_orig = df.iloc[:, -1:].values

print("Shape of Feature Matrix: ", x_orig.shape)
print("Shape Label Vector: ", y_orig.shape)

Shape of Feature Matrix:  (150, 3)
Shape Label Vector:  (150, 1)


Logistic Regression

In [136]:
X = df.iloc[:,0:-1]     # assign x to be all columns except 'species'
y = df.iloc[:,-1]       # assign y to be the 'species' column

"""
one hot encoding of the labels 
ex. since there are three classes, an instance belong to class 1 would have the label [1,0,0]
"""
encoder = OneHotEncoder(sparse=False)
X_encoded = encoder.fit_transform(X)

label_encoder = LabelEncoder()
y_onehot = encoder.fit_transform(y.values.reshape(-1, 1))



# split data into training and testing sets 
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.3, random_state=42) # test side: 30%, train 70%


# setting up tensorflow
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(len(X_train))

W = tf.Variable(tf.random.normal([X_train.shape[1], 3], dtype=tf.float64) * 0.01)
b = tf.Variable(tf.zeros([3], dtype=tf.float64))


# softmax regression model

"""
It first calculates the linear equation X∗W+b, 
and then applies the softmax function to convert these values into probabilities for each class.
"""
def model(X):
    return tf.nn.softmax(tf.matmul(X, W) + b)


# loss function
"""
This function calculates the Cross-Entropy loss. 
The Cross-Entropy loss is commonly used for classification problems as it measures the 
difference between two probability distributions - the true distribution (y_true) and 
the predicted distribution (y_pred).
"""

def loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float64)
    return -tf.reduce_sum(y_true * tf.math.log(y_pred))


# optimizer
optimizer = tf.optimizers.SGD(learning_rate=0.01)

c:\Users\mahsa\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\mahsa\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Training

In [137]:
for epoch in range(1000):
    for x_batch, y_batch in train_dataset:
        with tf.GradientTape() as tape:
            loss_val = loss(y_batch, model(x_batch))
            grads = tape.gradient(loss_val, [W, b])
            optimizer.apply_gradients(zip(grads, [W, b]))

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss_val.numpy()}")


Epoch 0, Loss: 115.46786375919454


Epoch 100, Loss: 52.55986540889124
Epoch 200, Loss: 18.290209167202296
Epoch 300, Loss: 16.734034109981195
Epoch 400, Loss: 15.38297178099784
Epoch 500, Loss: 14.207971529759956
Epoch 600, Loss: 13.21719279872391
Epoch 700, Loss: 12.422034114589104
Epoch 800, Loss: 11.812772734096212
Epoch 900, Loss: 11.353835136711249


Evaluation

In [138]:
y_pred = model(X_test)
predicted_class = tf.argmax(y_pred, axis=1)
actual_class = tf.argmax(y_test, axis=1)
accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted_class, actual_class), tf.float32))
print(f"Accuracy: {accuracy.numpy()}")


Accuracy: 0.9333333373069763
